# Colab + Google Drive Learning Cycle Template

**リポジトリ・データがすでに Drive にある前提**です。clone は不要です。

1. Mount Google Drive
2. Drive 上のリポジトリに移動し、依存関係をインストール
3. Drive 上の `data/raw` で学習・予測
4. 成果物（optuna_models, outputs など）を同じ Drive ツリーに保存


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Drive 上のリポジトリルート（Colab で Drive をマウントした後のパス）
# 例: マイドライブ/app/kyotei_Prediction なら '/content/drive/MyDrive/app/kyotei_Prediction'
PROJECT_DIR = '/content/drive/MyDrive/app/kyotei_Prediction'

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(f'Drive にリポジトリが見つかりません: {PROJECT_DIR}\nマウント後、上記 PROJECT_DIR をあなたのパスに合わせてください。')

os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
# requirements-colab: リポジトリ直下 or 親フォルダ(app) を探す
def find_req():
    for name in ('requirements-colab.txt', 'requirements-colab'):
        p = os.path.join(PROJECT_DIR, name)
        if os.path.isfile(p):
            return p
    parent = os.path.dirname(PROJECT_DIR)
    for name in ('requirements-colab.txt', 'requirements-colab'):
        p = os.path.join(parent, name)
        if os.path.isfile(p):
            return p
    return os.path.join(PROJECT_DIR, 'requirements.txt')
req_file = find_req()
print('Using', req_file)
!python -m pip install --upgrade pip -q
!pip install -r {req_file} -q


In [ ]:
import os

# データ・成果物のルート（リポジトリと同じツリーなら PROJECT_DIR のまま）
DRIVE_ROOT = PROJECT_DIR  # 別のフォルダにしたい場合はここを書き換え
# 学習データ: file=JSON の raw / db=SQLite
DATA_SOURCE = 'db'  # 'file' または 'db'
DATA_DIR = f'{DRIVE_ROOT}/kyotei_predictor/data/raw'
DB_PATH = f'{DRIVE_ROOT}/kyotei_predictor/data/kyotei_races.sqlite'  # data_source='db' のとき
YEAR_MONTH = '2025-01'
PREDICT_DATE = '2026-02-14'
N_TRIALS = 1
MINIMAL = True

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

print('DRIVE_ROOT =', DRIVE_ROOT)
print('DATA_SOURCE=', DATA_SOURCE)
print('DATA_DIR   =', DATA_DIR)
if DATA_SOURCE == 'db':
    print('DB_PATH    =', DB_PATH)


In [ ]:
# 取得データを Google Drive に保存する（RUN_FETCH=True で実行）
# 保存先: DATA_DIR = /content/drive/MyDrive/kyotei_prediction/data/raw

RUN_FETCH = False

if RUN_FETCH:
    !python -m kyotei_predictor.tools.batch.batch_fetch_all_venues \
      --start-date 2026-02-01 \
      --end-date 2026-02-14 \
      --stadiums ALL \
      --output-data-dir "{DATA_DIR}"
    print('取得データの保存先（Google Drive）:', DATA_DIR)
    if os.path.exists(DATA_DIR):
        n = len([f for r, d, files in os.walk(DATA_DIR) for f in files])
        print(f'  → 保存ファイル数: {n}')


In [ ]:
import subprocess

cmd = [
    'python',
    'scripts/run_colab_learning_cycle.py',
    '--drive-root', DRIVE_ROOT,
    '--data-source', DATA_SOURCE,
    '--data-dir', DATA_DIR,
    '--year-month', YEAR_MONTH,
    '--n-trials', str(N_TRIALS),
    '--predict-date', PREDICT_DATE,
]
if DATA_SOURCE == 'db' and DB_PATH:
    cmd += ['--db-path', DB_PATH]
if MINIMAL:
    cmd.append('--minimal')

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
import os

# 予測結果はローカル outputs/ に出力後、Drive へ同期済み
pred_path = f'outputs/predictions_{PREDICT_DATE}.json'
drive_pred_path = f'{DRIVE_ROOT}/outputs/predictions_{PREDICT_DATE}.json'
print('prediction file (local):', os.path.exists(pred_path), pred_path)
print('prediction file (Drive):', os.path.exists(drive_pred_path), drive_pred_path)

path = pred_path if os.path.exists(pred_path) else (drive_pred_path if os.path.exists(drive_pred_path) else None)
if path and os.path.exists(path):
    with open(path, 'r', encoding='utf-8') as f:
        pred = json.load(f)
    print('prediction_date:', pred.get('prediction_date'))
    print('summary:', pred.get('execution_summary'))
